# UCI Individual Household Electric Power Consumption + NASA POWER weather

Source: UCI ML Repository, `household_power_consumption.txt` (single household, Sceaux, France, minute-level, 2006-12 to 2010-11). Only 7 native columns, so enriched with matching local weather from the NASA POWER API (lat 48.778 / lon 2.290, same date range) -- a standard real-world enrichment for load forecasting (temperature/humidity drive heating and cooling demand), not synthetic augmentation.

Minute data is resampled to hourly: power/voltage/intensity by mean, the three sub-metering columns (already Wh per minute) by sum. Hours with fewer than 55/60 valid minutes are dropped rather than averaged from mostly-missing data.

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
import numpy as np
from common import report_candidate

HP_PATH = "../data/raw/household_power/household_power_consumption.txt"
NASA_PATH = "../data/raw/nasa_power_sceaux.csv"
PROCESSED_PATH = "../data/processed/uci_household_power_sceaux.csv"

hp = pd.read_csv(HP_PATH, sep=";", na_values=["?"], low_memory=False)
hp["timestamp"] = pd.to_datetime(hp["Date"] + " " + hp["Time"], format="%d/%m/%Y %H:%M:%S")
hp = hp.set_index("timestamp").drop(columns=["Date", "Time"])
print("minute-level missing %:")
print((hp.isna().mean() * 100).round(2))

In [ ]:
MEAN_COLS = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity"]
SUM_COLS = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

counts = hp["Global_active_power"].resample("1h").count()
valid_hours = counts[counts >= 55].index
print(f"keeping {len(valid_hours)}/{len(counts)} hours (>=55/60 valid minutes)")

hourly = pd.concat([
    hp[MEAN_COLS].resample("1h").mean(),
    hp[SUM_COLS].resample("1h").sum(),
], axis=1).loc[valid_hours]

hourly["apparent_power"] = np.sqrt(hourly.Global_active_power**2 + hourly.Global_reactive_power**2)
hourly["power_factor"] = hourly.Global_active_power / hourly.apparent_power.replace(0, np.nan)
hourly.shape

In [ ]:
with open(NASA_PATH, encoding="utf-8") as f:
    lines = f.readlines()
header_end = next(i for i, l in enumerate(lines) if "END HEADER" in l)

weather = pd.read_csv(NASA_PATH, skiprows=header_end + 1)
weather["timestamp"] = pd.to_datetime(dict(year=weather.YEAR, month=weather.MO, day=weather.DY, hour=weather.HR))
weather = weather.set_index("timestamp").drop(columns=["YEAR", "MO", "DY", "HR"]).replace(-999, np.nan)

df = hourly.join(weather, how="inner")
TARGET = "Global_active_power"
feature_cols = [c for c in df.columns if c != TARGET]
print(len(feature_cols), feature_cols)
(df.isna().mean() * 100).round(3)

In [ ]:
MAX_GAP_HOURS = 3
before = len(df)
df = df.interpolate(method="time", limit=MAX_GAP_HOURS).dropna(how="any")
print(f"dropped {before - len(df)} rows with unresolved gaps")

In [ ]:
report_candidate(df, TARGET, feature_cols, freq="1h", name="UCI Household Power + NASA weather, Sceaux FR (processed)")

In [ ]:
df.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={df.shape}")